# Notebook 01 — Data Loading & QA Dataset Creation

This notebook:
1. Downloads the Indiana University CXR dataset via Kaggle (images + CSV reports)
2. Parses the CSV reports (no XML parsing needed)
3. Generates a QA dataset using Groq LLaMA 3.1 8B Instant

**Before running:** Add these 4 keys to Colab Secrets (🔑 icon in left sidebar):
- `KAGGLE_USERNAME`, `KAGGLE_KEY` — from kaggle.com/settings → API → Create New Token
- `GROQ_API_KEY` — from console.groq.com
- `HF_TOKEN` — from huggingface.co/settings/tokens

In [ ]:
!pip install -q groq tqdm pandas kaggle

In [ ]:
import os, sys
from google.colab import drive, userdata
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/cxr_rag'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive mounted at', DRIVE_ROOT)

In [ ]:
# ── Kaggle credentials ────────────────────────────────────────────────────────
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
print('Kaggle credentials set.')

In [ ]:
# ── Download Indiana University CXR via Kaggle ────────────────────────────────
# Downloads: images/ folder + indiana_reports.csv + indiana_projections.csv
# Total size: ~1 GB

KAGGLE_DIR = '/content/openi'

if not os.path.exists(os.path.join(KAGGLE_DIR, 'indiana_reports.csv')):
    print('Downloading dataset (~1 GB) ...')
    import subprocess
    os.makedirs(KAGGLE_DIR, exist_ok=True)
    subprocess.run([
        'kaggle', 'datasets', 'download',
        '-d', 'raddar/chest-xrays-indiana-university',
        '-p', KAGGLE_DIR,
        '--unzip'
    ], check=True)
    print('Done.')
else:
    print('Dataset already downloaded.')

print('\nContents of', KAGGLE_DIR + ':')
for f in os.listdir(KAGGLE_DIR):
    print(' ', f)

In [ ]:
# ── Auto-detect image directory ───────────────────────────────────────────────
import glob

png_files = glob.glob(os.path.join(KAGGLE_DIR, '**', '*.png'), recursive=True)
if not png_files:
    raise RuntimeError('No PNG images found. Check the contents of ' + KAGGLE_DIR)

IMAGES_DIR = os.path.dirname(png_files[0])
print(f'Found {len(png_files)} PNG images in: {IMAGES_DIR}')

In [ ]:
# ── Clone project repo ────────────────────────────────────────────────────────
REPO_URL = 'https://github.com/mohamedtaha77/cxr-rag-system.git'

if not os.path.exists('/content/cxr-rag-system'):
    !git clone {REPO_URL} /content/cxr-rag-system
else:
    !git -C /content/cxr-rag-system pull

sys.path.insert(0, '/content/cxr-rag-system')
print('Repo ready.')

In [ ]:
# ── Load dataset from Kaggle CSVs ─────────────────────────────────────────────
# Uses indiana_reports.csv + indiana_projections.csv
# Much simpler than XML parsing — projections CSV tells us which images are frontal

from src.data.openi_loader import OpenILoader

loader = OpenILoader(images_dir=IMAGES_DIR)
df = loader.load_from_kaggle_csvs(kaggle_dir=KAGGLE_DIR)

print(f'Loaded {len(df)} studies with impression + frontal image')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

In [ ]:
# ── Train / val / test split ──────────────────────────────────────────────────
import pandas as pd, shutil

train_df, val_df, test_df = loader.train_val_test_split(df)
full_df = pd.concat([train_df, val_df, test_df])

os.makedirs('/content/cxr-rag-system/data/processed', exist_ok=True)
full_df.to_csv('/content/cxr-rag-system/data/processed/reports_corpus.csv', index=False)
shutil.copy('/content/cxr-rag-system/data/processed/reports_corpus.csv',
            os.path.join(DRIVE_ROOT, 'reports_corpus.csv'))

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print('Sample impressions:')
for imp in full_df['impression'].head(3):
    print(' -', imp[:120])

In [ ]:
# ── Configure Groq ────────────────────────────────────────────────────────────
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

from src.data.qa_creator import QACreator
creator = QACreator(groq_api_key=GROQ_API_KEY)
print('QA creator ready.')

In [ ]:
# ── Generate QA dataset ───────────────────────────────────────────────────────
# max_studies=500  → ~3,000 pairs, ~30 min
# max_studies=None → full dataset (~23,000 pairs, ~3 hrs)

QA_OUTPUT = '/content/cxr-rag-system/data/processed/qa_dataset.jsonl'

pairs = creator.generate_dataset(
    df=full_df,
    output_path=QA_OUTPUT,
    max_studies=500,
)

print(f'Generated {len(pairs)} QA pairs')
shutil.copy(QA_OUTPUT, os.path.join(DRIVE_ROOT, 'qa_dataset.jsonl'))

In [ ]:
# ── Inspect sample QA pairs ───────────────────────────────────────────────────
import json

with open(QA_OUTPUT) as f:
    samples = [json.loads(l) for l in f][:5]

for s in samples:
    print(f"Category : {s['category']}")
    print(f"Q        : {s['question']}")
    print(f"A        : {s['answer']}")
    print()

In [ ]:
# ── Dataset statistics ────────────────────────────────────────────────────────
qa_df = pd.read_json(QA_OUTPUT, lines=True)
print(f'Total pairs   : {len(qa_df)}')
print(f'Unique studies: {qa_df["study_id"].nunique()}')
print(f'\nSplit distribution:')
print(qa_df['split'].value_counts())
print(f'\nCategory distribution:')
print(qa_df['category'].value_counts())